# Task-Aware Joint-Action PPO Scheduler

PPO scheduler for reliability-aware task assignment with task-aware observations and
masked joint actions over processor core, voltage/frequency level, and standby-replica
count. Training and evaluation outputs are stored under `artifacts/`.


## Imports


In [ ]:
import os
from pathlib import Path
import math
import time
import random
import csv
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


def project_root() -> Path:
    """Find the repository root from a notebook or project subdirectory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "requirements.txt").exists() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Repository root not found")


PROJECT_ROOT = project_root()


## Task-aware environment


In [ ]:
class Envir:
    """Task-aware scheduling environment.

    Key modeling choices:
    - Observation contains per-core utilization + normalized features of the CURRENT task.
    - The environment owns the episode task order, so the returned observation always matches
      the task that the next action will schedule.
    - A joint action selects (primary_core, VF level, replica setting).
    - Replica semantics are STANDBY/backup semantics: replicas do not add normal CPU utilization
      merely because they exist. The primary task alone updates the utilization state.
    - Replica count affects reliability and the load-dependent replica penalty. That
      penalty is the model's abstraction for the increasing cost/overlap pressure of deferred
      backup scheduling near deadlines at high load.
    - Capacity-infeasible primary placements are masked before sampling.

    Model scope:
    The assignment model does not include an event timeline, fault-arrival trace, or explicit
    replica start times. The assignment environment therefore does not infer per-replica execution events.
    Fault-triggered backup execution and deadline-window overlap require explicit timing events.
    Replica pressure is represented by a load-dependent penalty rather than by permanently
    charging replica utilization.
    """

    TASK_FEATURE_KEYS = (
        "initial_exec_task_time",
        "deadlines",
        "util",
        "peak",
        "alpha_C",
        "I0_exp_term",
        "base_util",
    )

    def __init__(
        self,
        initial_voltage,
        initial_frequency,
        cores,
        main_tasks,
        alpha,
        util_load,
        capacity_limit=1.1,
        tsp=1.5,
        rep_max=10,
    ):
        self.initial_voltage = float(initial_voltage)
        self.initial_frequency = float(initial_frequency)
        self.cores = int(cores)
        self.main_tasks = int(main_tasks)
        self.alpha = tuple(alpha)
        self.load = float(util_load)
        self.capacity_limit = float(capacity_limit)
        self.TSP = float(tsp)
        self.rep_max = int(rep_max)
        self.threshold_load = 0.7

        self.vf_levels = {
            0: {"frequency": 800e6, "voltage": 0.9},
            1: {"frequency": 1.1e9, "voltage": 0.95},
            2: {"frequency": 1.4e9, "voltage": 1.0},
            3: {"frequency": 1.7e9, "voltage": 1.05},
            4: {"frequency": 2.0e9, "voltage": 1.1},
        }
        self.n_vf = len(self.vf_levels)
        self.n_rep = self.rep_max + 1
        self.n_actions = self.cores * self.n_vf * self.n_rep

        # Reliability parameters for the assignment model.
        self.lambda_0 = 1e-7
        self.Delta = 1.0
        self.V_max = self.vf_levels[max(self.vf_levels)]["voltage"]

        self.data: List[Dict] = []
        self.train_data: List[Dict] = []
        self.feature_min: Optional[np.ndarray] = None
        self.feature_max: Optional[np.ndarray] = None

        self.state = np.zeros(self.cores, dtype=np.float32)
        self.system_tasks: List[float] = []
        self.task_order: List[int] = []
        self.task_cursor = 0
        self.current_task_idx: Optional[int] = None
        self.rng = np.random.default_rng()

        # core utilizations + current-task features + [episode_progress, running_reliability]
        self.obs_dim = self.cores + len(self.TASK_FEATURE_KEYS) + 2

    # ------------------------------------------------------------------
    # Domain helpers
    # ------------------------------------------------------------------
    def fault_rate(self, V):
        return self.lambda_0 * 10 ** ((self.V_max - V) / self.Delta)

    def task_reliability(self, t, V):
        return float(np.exp(-self.fault_rate(V) * t))

    def total_task_reliability(self, task_tuple):
        n, exec_time, voltage = task_tuple
        n = int(n)
        if n <= 0:
            return 0.0
        single = self.task_reliability(exec_time, voltage)
        # Reliability of at least one successful copy.
        return float(1.0 - (1.0 - single) ** n)

    def system_reliability(self, task_total_reliabilities):
        if not task_total_reliabilities:
            return 1.0
        return float(np.mean(task_total_reliabilities))

    def power_sharp(self, x, n=1):
        return float(2 * (x ** n) - 1)

    # ------------------------------------------------------------------
    # Data and feature normalization
    # ------------------------------------------------------------------
    def read_data(self, input_files):
        """Load one task from the first row of each CSV, using the first row of each task CSV."""
        self.data = []
        for file in input_files:
            task_name = os.path.basename(file).replace(".csv", "")
            task_data = pd.read_csv(file)
            if task_data.empty:
                raise ValueError(f"Task CSV is empty: {file}")

            row = task_data.iloc[0]
            required = ["#", "static", "avg_power", "alpha_C", "I0_exp_term", "util", "peak", "deadline", "min"]
            missing = [c for c in required if c not in task_data.columns]
            if missing:
                raise ValueError(f"Missing columns in {file}: {missing}")

            initial_exec_task_time = float(row["#"])
            deadline = float(row["deadline"])
            if deadline <= 0:
                raise ValueError(f"deadline must be positive in {file}")

            self.data.append({
                "name": task_name,
                "alpha_C": float(row["alpha_C"]),
                "I0_exp_term": float(row["I0_exp_term"]),
                "initial_exec_task_time": initial_exec_task_time,
                "deadlines": deadline,
                "util": float(row["util"]),
                "peak": float(row["peak"]),
                "min": float(row["min"]),
                "power": float(row["avg_power"]),
                "static": float(row["static"]),
                "base_util": initial_exec_task_time / deadline,
            })

        self.train_data = list(self.data)
        if not self.train_data:
            raise ValueError("No tasks were loaded.")
        self.main_tasks = len(self.train_data)
        self._fit_task_normalizer()
        return self.train_data

    def _raw_task_features(self, task_idx):
        t = self.train_data[int(task_idx)]
        return np.array([t[k] for k in self.TASK_FEATURE_KEYS], dtype=np.float32)

    def _fit_task_normalizer(self):
        x = np.stack([self._raw_task_features(i) for i in range(len(self.train_data))])
        self.feature_min = np.min(x, axis=0)
        self.feature_max = np.max(x, axis=0)

    def normalized_task_features(self, task_idx):
        if self.feature_min is None or self.feature_max is None:
            raise RuntimeError("Call read_data() before requesting observations.")
        x = self._raw_task_features(task_idx)
        span = self.feature_max - self.feature_min
        safe_span = np.where(span > 1e-12, span, 1.0)
        z = (x - self.feature_min) / safe_span
        z = np.where(span > 1e-12, z, 0.0)
        return np.clip(z, 0.0, 1.0).astype(np.float32)

    def print_task_details(self):
        for idx, task in enumerate(self.train_data):
            peak_i = self.peak_power(idx, vf_select=max(self.vf_levels))
            print(
                f"Task {idx:02d} | {task['name']} | baseline/max-VF peak={peak_i:.6f} "
                f"| deadline={task['deadlines']} | exec={task['initial_exec_task_time']} "
                f"| base_util={task['base_util']:.4f}"
            )
        print(f"\nTotal tasks: {len(self.train_data)}")

    # ------------------------------------------------------------------
    # Observation / episode management
    # ------------------------------------------------------------------
    def _running_reliability(self):
        return self.system_reliability(self.system_tasks)

    def _observation_for(self, task_idx):
        core_part = np.clip(self.state / self.capacity_limit, 0.0, 1.5).astype(np.float32)
        task_part = self.normalized_task_features(task_idx)
        progress = float(self.task_cursor / max(1, len(self.task_order)))
        context = np.array([progress, self._running_reliability()], dtype=np.float32)
        return np.concatenate([core_part, task_part, context]).astype(np.float32)

    def _terminal_observation(self):
        core_part = np.clip(self.state / self.capacity_limit, 0.0, 1.5).astype(np.float32)
        task_part = np.zeros(len(self.TASK_FEATURE_KEYS), dtype=np.float32)
        context = np.array([1.0, self._running_reliability()], dtype=np.float32)
        return np.concatenate([core_part, task_part, context]).astype(np.float32)

    def reset(self, *, seed=None, shuffle_tasks=True):
        if not self.train_data:
            raise RuntimeError("No task data loaded. Call read_data() first.")
        if seed is not None:
            self.rng = np.random.default_rng(seed)

        self.state = np.zeros(self.cores, dtype=np.float32)
        self.system_tasks = []
        self.task_order = list(range(len(self.train_data)))
        if shuffle_tasks:
            self.rng.shuffle(self.task_order)
        self.task_cursor = 0
        self.current_task_idx = self.task_order[0]
        return self._observation_for(self.current_task_idx), {
            "task": int(self.current_task_idx),
            "task_name": self.train_data[self.current_task_idx]["name"],
        }

    # ------------------------------------------------------------------
    # Joint action encoding and projection
    # ------------------------------------------------------------------
    def encode_action(self, core_select, vf_select, replica_n):
        core_select = int(core_select)
        vf_select = int(vf_select)
        replica_n = int(replica_n)
        if not (0 <= core_select < self.cores):
            raise ValueError("Invalid core index")
        if not (0 <= vf_select < self.n_vf):
            raise ValueError("Invalid VF index")
        if not (0 <= replica_n <= self.rep_max):
            raise ValueError("Invalid replica count")
        return ((core_select * self.n_vf) + vf_select) * self.n_rep + replica_n

    def decode_action(self, action_idx):
        action_idx = int(action_idx)
        if not (0 <= action_idx < self.n_actions):
            raise ValueError(f"action_idx must be in [0, {self.n_actions})")
        replica_n = action_idx % self.n_rep
        q = action_idx // self.n_rep
        vf_select = q % self.n_vf
        core_select = q // self.n_vf
        return int(core_select), int(vf_select), int(replica_n)

    def task_utilization(self, task_idx, vf_select):
        t = self.train_data[int(task_idx)]
        f = self.vf_levels[int(vf_select)]["frequency"]
        return float((t["initial_exec_task_time"] / (f / self.initial_frequency)) / t["deadlines"])

    def peak_power(self, task_idx, vf_select):
        t = self.train_data[int(task_idx)]
        vf = self.vf_levels[int(vf_select)]
        f, v = vf["frequency"], vf["voltage"]
        p_static = t["I0_exp_term"] * v
        return float(t["peak"] * (v ** 2) * f + p_static)

    def project_action(self, action_idx, task_idx=None):
        """Project only the PRIMARY execution onto utilization.

        `replica_n` is a standby reliability decision. It does not multiply current utilization.
        Replica-related execution pressure at high load is represented by the load-dependent
        penalty in `_reliability_terms()` until an explicit timing/fault model is available.
        """
        if task_idx is None:
            if self.current_task_idx is None:
                raise RuntimeError("No current task.")
            task_idx = self.current_task_idx

        core_select, vf_select, replica_n = self.decode_action(action_idx)
        util_inc = self.task_utilization(task_idx, vf_select)

        projected = self.state.astype(np.float64).copy()
        projected[core_select] += util_inc

        return {
            "core": core_select,
            "vf": vf_select,
            "replica_n": replica_n,
            "util_inc": util_inc,
            "projected_state": projected.astype(np.float32),
            "placements": (int(core_select),),
            "primary_resource_copies": 1,
            "replica_execution_mode": "standby",
            "capacity_violation": bool(np.max(projected) > self.capacity_limit + 1e-12),
            "thermal_violation": bool(self.peak_power(task_idx, vf_select) > self.TSP),
        }

    def joint_action_mask(self, task_idx=None):
        """True means the PRIMARY placement is capacity-feasible for the current task.

        Standby replicas are not charged as always-running utilization. Thermal
        safety stays in the reward rather than the hard mask. If every primary placement violates
        capacity, the least-overloading actions are exposed instead of returning an all-False mask.
        """
        if task_idx is None:
            task_idx = self.current_task_idx
        mask = np.zeros(self.n_actions, dtype=np.bool_)
        max_loads = np.empty(self.n_actions, dtype=np.float32)

        for a in range(self.n_actions):
            p = self.project_action(a, task_idx)
            max_loads[a] = float(np.max(p["projected_state"]))
            mask[a] = not p["capacity_violation"]

        if not np.any(mask):
            best = float(np.min(max_loads))
            mask = np.isclose(max_loads, best, rtol=0.0, atol=1e-8)
        return mask

    # ------------------------------------------------------------------
    # Reward
    # ------------------------------------------------------------------
    def _utilization_reward(self, action_idx, task_idx):
        """Reward primary-core choice relative to alternatives for the same VF/replica decision."""
        core_select, vf_select, replica_n = self.decode_action(action_idx)
        candidate_max = []
        chosen_max = None
        for c in range(self.cores):
            a = self.encode_action(c, vf_select, replica_n)
            p = self.project_action(a, task_idx)
            m = float(np.max(p["projected_state"]))
            candidate_max.append(m)
            if c == core_select:
                chosen_max = m

        best = min(candidate_max)
        worst = max(candidate_max)
        if worst - best <= 1e-8:
            return 1.0
        r = 1.0 - 2.0 * (chosen_max - best) / (worst - best)
        return float(np.clip(r, -1.0, 1.0))

    def _reliability_terms(self, task_idx, vf_select, replica_n):
        """Reliability benefit + standby-replica scheduling pressure.

        Replicas are NOT treated as continuously executing copies. Their reliability benefit is
        analytical, while their scheduling cost is represented by the load-dependent
        penalty: cheap at low load, increasingly expensive once the system enters the high-load
        region where deferred backup windows near deadlines are more likely to overlap/compete.

        Fault-triggered execution and overlap duration require an event/fault timeline, which
        is outside the assignment model.
        """
        t = self.train_data[int(task_idx)]
        f = self.vf_levels[int(vf_select)]["frequency"]
        v = self.vf_levels[int(vf_select)]["voltage"]

        # Keep fractional execution time before scaling.
        exec_time_scaled = float(t["initial_exec_task_time"] / (f / self.initial_frequency)) * 1000.0
        task_rel = self.total_task_reliability((int(replica_n), exec_time_scaled, float(v)))
        prospective = self.system_tasks + [task_rel]
        system_r = self.system_reliability(prospective)
        reliability_reward = self.power_sharp(system_r)

        replica_penalty = 0.0
        if replica_n <= 1:
            reliability_reward = -1.0
        elif self.load < self.threshold_load:
            replica_penalty = (
                (1 / self.cores)
                * (self.load ** 4)
                * (2 ** replica_n)
            ) / (2 ** self.rep_max)
            reliability_reward -= replica_penalty
        else:
            # Apply the high-load penalty with numerical protection.
            denom = max(1e-6, 1.1 - self.load)
            replica_penalty = (
                ((1 / denom) ** 2)
                * (2 ** replica_n)
                * (1 / self.cores)
                * (replica_n ** 2)
            ) / (2 ** self.rep_max)
            reliability_reward -= replica_penalty

        reliability_reward = float(np.clip(reliability_reward, -1.0, 1.0))
        return task_rel, system_r, reliability_reward, float(replica_penalty)

    def reward_function(self, action_idx, task_idx):
        core_select, vf_select, replica_n = self.decode_action(action_idx)
        projection = self.project_action(action_idx, task_idx)

        peak = self.peak_power(task_idx, vf_select)
        f = self.vf_levels[vf_select]["frequency"]
        f_min = self.vf_levels[0]["frequency"]
        f_max = self.vf_levels[self.n_vf - 1]["frequency"]
        f_norm = (f - f_min) / (f_max - f_min + 1e-12)
        thermal_reward = -1.0 if peak > self.TSP else float(1.0 - f_norm)

        utilization_reward = self._utilization_reward(action_idx, task_idx)
        task_rel, system_r, reliability_reward, replica_penalty = self._reliability_terms(task_idx, vf_select, replica_n)
        projection["replica_penalty"] = float(replica_penalty)

        alpha1, alpha2, alpha3, alpha4 = self.alpha
        # alpha2 is unused because the reward has no separate performance term.
        total_reward = (
            alpha1 * thermal_reward
            + alpha3 * utilization_reward
            + alpha4 * reliability_reward
        )

        # Capacity is treated as a hard scheduling constraint. Normally masked actions prevent this;
        # this branch handles the rare all-actions-infeasible fallback state.
        if projection["capacity_violation"]:
            total_reward = -1.0

        total_reward = float(np.clip(total_reward, -1.0, 1.0))
        weighted_terms = (
            alpha1 * thermal_reward,
            alpha3 * utilization_reward,
            alpha4 * reliability_reward,
        )
        return total_reward, projection, task_rel, system_r, weighted_terms

    def step(self, action_idx):
        if self.current_task_idx is None:
            raise RuntimeError("Episode is finished. Call reset().")

        task_idx = int(self.current_task_idx)
        task_name = self.train_data[task_idx]["name"]
        reward, projection, task_rel, system_r, weighted_terms = self.reward_function(action_idx, task_idx)

        # Apply only the PRIMARY task utilization. Standby replicas do not automatically consume
        # utilization; their high-load scheduling pressure is handled in the reliability penalty.
        self.state = projection["projected_state"].astype(np.float32)
        self.system_tasks.append(float(task_rel))

        self.task_cursor += 1
        terminated = self.task_cursor >= len(self.task_order)
        if terminated:
            self.current_task_idx = None
            next_obs = self._terminal_observation()
        else:
            self.current_task_idx = int(self.task_order[self.task_cursor])
            next_obs = self._observation_for(self.current_task_idx)

        core_select, vf_select, replica_n = self.decode_action(action_idx)
        info = {
            "task": task_idx,
            "task_name": task_name,
            "action_idx": int(action_idx),
            "core_select": core_select,
            "vf_select": vf_select,
            "replica_n": replica_n,
            "primary_resource_copies": projection["primary_resource_copies"],
            "replica_execution_mode": projection["replica_execution_mode"],
            "placements": tuple(projection["placements"]),
            "primary_util_inc": float(projection["util_inc"]),
            "replica_penalty": float(projection["replica_penalty"]),
            "projected_state": projection["projected_state"].copy(),
            "capacity_violation": projection["capacity_violation"],
            "thermal_violation": projection["thermal_violation"],
            "task_reliability": float(task_rel),
            "system_reliability": float(system_r),
            "reward_terms": tuple(float(x) for x in weighted_terms),
        }
        return next_obs, float(reward), bool(terminated), False, info


## Joint-action policy


In [ ]:
class PolicyNet(nn.Module):
    """Shared backbone with one JOINT action head and one value head."""

    def __init__(self, obs_dim, n_actions, hidden=256):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.action_head = nn.Linear(hidden, n_actions)
        self.value_head = nn.Linear(hidden, 1)

    def forward(self, x):
        z = self.backbone(x)
        logits = self.action_head(z)
        value = self.value_head(z).squeeze(-1)
        return logits, value


def masked_categorical(logits, mask):
    if mask.dtype != torch.bool:
        mask = mask.bool()
    if mask.shape != logits.shape:
        raise ValueError(f"mask shape {mask.shape} does not match logits shape {logits.shape}")
    if torch.any(~torch.any(mask, dim=-1) if mask.ndim > 1 else ~torch.any(mask)):
        raise ValueError("Action mask contains an all-False row.")
    neg = torch.finfo(logits.dtype).min
    masked_logits = torch.where(mask, logits, torch.full_like(logits, neg))
    return torch.distributions.Categorical(logits=masked_logits)


## PPO training and evaluation


In [ ]:
class PPOAgent:
    def __init__(
        self,
        env: Envir,
        lr=3e-4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_eps=0.2,
        vf_coef=0.5,
        ent_coef=0.01,
        update_epochs=4,
        minibatch_size=1024,
        device=None,
    ):
        self.env = env
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.policy = PolicyNet(env.obs_dim, env.n_actions).to(self.device)
        self.optim = optim.Adam(self.policy.parameters(), lr=lr)

        self.gamma = float(gamma)
        self.lam = float(gae_lambda)
        self.clip_eps = float(clip_eps)
        self.vf_coef = float(vf_coef)
        self.ent_coef = float(ent_coef)
        self.update_epochs = int(update_epochs)
        self.minibatch_size = int(minibatch_size)

    @torch.no_grad()
    def act(self, obs, action_mask, deterministic=False):
        self.policy.eval()
        x = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        mask_t = torch.as_tensor(action_mask, dtype=torch.bool, device=self.device)
        logits, value = self.policy(x)
        dist = masked_categorical(logits, mask_t)
        action = torch.argmax(dist.logits) if deterministic else dist.sample()
        logp = dist.log_prob(action)
        return int(action.item()), float(logp.item()), float(value.item())

    @torch.no_grad()
    def value(self, obs):
        x = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        _, v = self.policy(x)
        return float(v.item())

    def _compute_gae(self, rewards, values, dones, next_value):
        T = len(rewards)
        adv = np.zeros(T, dtype=np.float32)
        last_gae = 0.0
        for t in reversed(range(T)):
            nonterminal = 1.0 - float(dones[t])
            next_v = next_value if t == T - 1 else values[t + 1]
            delta = rewards[t] + self.gamma * next_v * nonterminal - values[t]
            last_gae = delta + self.gamma * self.lam * nonterminal * last_gae
            adv[t] = last_gae
        return adv, adv + values

    def update(self, batch):
        self.policy.train()
        obs = torch.as_tensor(np.stack(batch["obs"]), dtype=torch.float32, device=self.device)
        masks = torch.as_tensor(np.stack(batch["mask"]), dtype=torch.bool, device=self.device)
        actions = torch.as_tensor(batch["action"], dtype=torch.long, device=self.device)
        old_logp = torch.as_tensor(batch["logp"], dtype=torch.float32, device=self.device)
        returns = torch.as_tensor(batch["ret"], dtype=torch.float32, device=self.device)
        adv = torch.as_tensor(batch["adv"], dtype=torch.float32, device=self.device)

        if adv.numel() > 1:
            adv = (adv - adv.mean()) / (adv.std(unbiased=False) + 1e-8)

        N = obs.size(0)
        idxs = np.arange(N)
        metrics = []

        for _ in range(self.update_epochs):
            np.random.shuffle(idxs)
            for start in range(0, N, self.minibatch_size):
                mb_idx_np = idxs[start:start + self.minibatch_size]
                mb_idx = torch.as_tensor(mb_idx_np, dtype=torch.long, device=self.device)

                logits, values = self.policy(obs[mb_idx])
                dist = masked_categorical(logits, masks[mb_idx])
                logp = dist.log_prob(actions[mb_idx])
                entropy = dist.entropy().mean()

                ratio = torch.exp(logp - old_logp[mb_idx])
                surr1 = ratio * adv[mb_idx]
                surr2 = torch.clamp(ratio, 1.0 - self.clip_eps, 1.0 + self.clip_eps) * adv[mb_idx]
                policy_loss = -torch.min(surr1, surr2).mean()
                value_loss = 0.5 * (returns[mb_idx] - values).pow(2).mean()
                loss = policy_loss + self.vf_coef * value_loss - self.ent_coef * entropy

                self.optim.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
                self.optim.step()

                metrics.append((float(policy_loss.item()), float(value_loss.item()), float(entropy.item())))

        if not metrics:
            return {"policy_loss": np.nan, "value_loss": np.nan, "entropy": np.nan}
        m = np.asarray(metrics, dtype=np.float64)
        return {"policy_loss": m[:, 0].mean(), "value_loss": m[:, 1].mean(), "entropy": m[:, 2].mean()}

    def train(
        self,
        time_elapse,
        episode_elapse,
        csv_dir,
        tasks_n,
        episodes=1600,
        save_dir="ppo_results",
        policy_dir="policy",
        log_last_k=100,
        seed=42,
        alphas=(0.45, 0.0, 0.3, 0.25),
    ):
        os.makedirs(save_dir, exist_ok=True)
        os.makedirs(policy_dir, exist_ok=True)
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        csv_files = [os.path.join(csv_dir, f"train_task{j}.csv") for j in range(1, tasks_n + 1)]
        self.env.alpha = tuple(alphas)
        self.env.read_data(csv_files)
        self.env.print_task_details()

        rewards_hist = []
        detail_log_file_path = os.path.join(save_dir, "detail_tasks_after_learn2.txt")
        open(detail_log_file_path, "w").close()

        for ep in range(1, episodes + 1):
            ep_start = time.time()
            obs, _ = self.env.reset(seed=(seed if ep == 1 else None), shuffle_tasks=True)
            terminated = False
            ep_rewards = 0.0

            traj_obs, traj_mask, traj_action = [], [], []
            traj_logp, traj_val, traj_rew, traj_done = [], [], [], []
            step_infos = []

            while not terminated:
                step_start = time.time()
                action_mask = self.env.joint_action_mask()
                action_idx, logp, value = self.act(obs, action_mask)
                next_obs, reward, terminated, truncated, info = self.env.step(action_idx)
                done = terminated or truncated

                traj_obs.append(obs.copy())
                traj_mask.append(action_mask.copy())
                traj_action.append(action_idx)
                traj_logp.append(logp)
                traj_val.append(value)
                traj_rew.append(reward)
                traj_done.append(done)
                step_infos.append(info)

                ep_rewards += reward
                obs = next_obs

                elapsed = time.time() - step_start
                worst_time = time_elapse[0]
                avg_time, n_time = time_elapse[1]
                time_elapse[0] = max(worst_time, elapsed)
                time_elapse[1] = ((avg_time * n_time + elapsed) / (n_time + 1), n_time + 1)

            next_value = 0.0 if traj_done[-1] else self.value(obs)
            values_np = np.asarray(traj_val, dtype=np.float32)
            rewards_np = np.asarray(traj_rew, dtype=np.float32)
            dones_np = np.asarray(traj_done, dtype=np.float32)
            adv, ret = self._compute_gae(rewards_np, values_np, dones_np, next_value)

            batch = {
                "obs": traj_obs,
                "mask": traj_mask,
                "action": np.asarray(traj_action, dtype=np.int64),
                "logp": np.asarray(traj_logp, dtype=np.float32),
                "ret": ret.astype(np.float32),
                "adv": adv.astype(np.float32),
            }
            update_stats = self.update(batch)

            rewards_hist.append(ep_rewards / max(1, len(traj_rew)))
            ep_elapsed = time.time() - ep_start
            worst_ep = episode_elapse[0]
            avg_ep, n_ep = episode_elapse[1]
            episode_elapse[0] = max(worst_ep, ep_elapsed)
            episode_elapse[1] = ((avg_ep * n_ep + ep_elapsed) / (n_ep + 1), n_ep + 1)

            if ep % 100 == 0 or ep == 1 or ep == episodes:
                print(
                    f"[PPO] Episode {ep}/{episodes} | avg_step_reward={rewards_hist[-1]:.4f} "
                    f"| policy_loss={update_stats['policy_loss']:.4f} "
                    f"| value_loss={update_stats['value_loss']:.4f}"
                )

            if ep > episodes - log_last_k:
                with open(detail_log_file_path, "a") as log_file:
                    for i_step, info in enumerate(step_infos):
                        loads = "[" + ",".join(f"{u:.4f}" for u in info["projected_state"]) + "]"
                        log_file.write(
                            f"Episode {ep}, step {i_step}, task={info['task']} ({info['task_name']}), "
                            f"action={info['action_idx']}, core={info['core_select']}, vf={info['vf_select']}, "
                            f"rep={info['replica_n']}, placements={info['placements']}, "
                            f"Reward={traj_rew[i_step]:.6f}, ProjectedUtil={loads}, "
                            f"TaskRel={info['task_reliability']:.8f}, SystemRel={info['system_reliability']:.8f}, "
                            f"CapacityViolation={info['capacity_violation']}, ThermalViolation={info['thermal_violation']}\n"
                        )

        torch.save(
            {
                "model_state_dict": self.policy.state_dict(),
                "obs_dim": self.env.obs_dim,
                "n_actions": self.env.n_actions,
                "feature_min": self.env.feature_min,
                "feature_max": self.env.feature_max,
                "task_feature_keys": self.env.TASK_FEATURE_KEYS,
            },
            os.path.join(policy_dir, "policy.pt"),
        )

        if len(rewards_hist) >= 2:
            window_size = min(len(rewards_hist), max(5, int(episodes * 0.06)))
            kernel = np.ones(window_size) / window_size
            smoothed = np.convolve(rewards_hist, kernel, mode="valid")
            plt.figure(figsize=(12, 6))
            plt.plot(smoothed, label=f"Smoothed Rewards (window={window_size})")
            plt.xlabel("Episodes")
            plt.ylabel("Reward per step")
            plt.title("Task-aware joint-action PPO")
            plt.legend()
            plt.grid()
            plt.savefig(os.path.join(save_dir, "smoothed_rewards_PPO.png"), dpi=300, bbox_inches="tight")
            plt.close()

        return rewards_hist

    @torch.no_grad()
    def evaluate(self, episodes=20, seed=1000, deterministic=True):
        rows = []
        for ep in range(episodes):
            obs, _ = self.env.reset(seed=seed + ep, shuffle_tasks=True)
            done = False
            while not done:
                mask = self.env.joint_action_mask()
                action_idx, _, _ = self.act(obs, mask, deterministic=deterministic)
                obs, reward, terminated, truncated, info = self.env.step(action_idx)
                done = terminated or truncated
                rows.append({
                    "episode": ep,
                    "reward": reward,
                    "task": info["task"],
                    "core": info["core_select"],
                    "vf": info["vf_select"],
                    "replica_n": info["replica_n"],
                    "replica_penalty": info["replica_penalty"],
                    "task_reliability": info["task_reliability"],
                    "system_reliability": info["system_reliability"],
                    "capacity_violation": info["capacity_violation"],
                    "thermal_violation": info["thermal_violation"],
                    "max_core_util": float(np.max(info["projected_state"])),
                })
        df = pd.DataFrame(rows)
        summary = {
            "mean_reward": float(df["reward"].mean()),
            "capacity_violation_rate": float(df["capacity_violation"].mean()),
            "thermal_violation_rate": float(df["thermal_violation"].mean()),
            "mean_task_reliability": float(df["task_reliability"].mean()),
            "mean_replica_n": float(df["replica_n"].mean()),
            "mean_replica_penalty": float(df["replica_penalty"].mean()),
            "mean_max_core_util": float(df["max_core_util"].mean()),
        }
        return summary, df


## Self-test


In [ ]:
# Lightweight self-test: no project dataset is needed.
# It verifies task-awareness, joint actions, standby-replica semantics,
# action masking, stepping, and a tiny PPO update.

import tempfile

_tmp = tempfile.mkdtemp(prefix="nnppo_smoke_")

_synthetic = [
    # exec, deadline, util, peak, alpha_C, I0
    (0.20, 1.0, 0.20, 2.0e-10, 2.0e-10, 0.020),
    (0.45, 1.2, 0.38, 3.0e-10, 3.0e-10, 0.030),
    (0.15, 0.6, 0.25, 1.0e-10, 1.0e-10, 0.010),
    (0.30, 0.9, 0.33, 2.5e-10, 2.5e-10, 0.025),
]

for j, (ex, dl, util, peak, alpha_c, i0) in enumerate(_synthetic, start=1):
    pd.DataFrame([{
        "#": ex,
        "static": 0.1,
        "avg_power": 1.0,
        "alpha_C": alpha_c,
        "I0_exp_term": i0,
        "util": util,
        "peak": peak,
        "deadline": dl,
        "min": 0.0,
    }]).to_csv(os.path.join(_tmp, f"train_task{j}.csv"), index=False)

_test_env = Envir(
    initial_voltage=1.1,
    initial_frequency=2e9,
    cores=4,
    main_tasks=4,
    alpha=(0.3, 0.0, 0.55, 0.15),
    util_load=0.4,
)
_test_env.read_data([os.path.join(_tmp, f"train_task{j}.csv") for j in range(1, 5)])

_obs0, _ = _test_env.reset(seed=7, shuffle_tasks=False)
_obs1_same_system = _test_env._observation_for(1)

assert _obs0.shape == (_test_env.obs_dim,)
assert not np.allclose(_obs0, _obs1_same_system), "Task-aware observation test failed."

for _tpl in [(0, 0, 0), (3, 4, 10), (2, 1, 4)]:
    _a = _test_env.encode_action(*_tpl)
    assert _test_env.decode_action(_a) == _tpl

# Standby-replica utilization check:
# for identical core + VF, changing replica_n must NOT change normal utilization.
_a_rep2 = _test_env.encode_action(0, 4, 2)
_a_rep8 = _test_env.encode_action(0, 4, 8)
_p2 = _test_env.project_action(_a_rep2, task_idx=0)
_p8 = _test_env.project_action(_a_rep8, task_idx=0)
assert np.allclose(_p2["projected_state"], _p8["projected_state"]), (
    "Standby replicas must not multiply normal CPU utilization."
)
assert _p2["replica_execution_mode"] == "standby"
assert _p8["replica_execution_mode"] == "standby"

# Capacity feasibility for the same core/VF is likewise replica-count independent
# because only the primary execution is currently active.
_mask = _test_env.joint_action_mask(task_idx=0)
assert _mask.shape == (_test_env.n_actions,)
assert np.any(_mask)
assert _mask[_a_rep2] == _mask[_a_rep8]

# Replica cost still becomes much stronger in the high-load region.
_saved_load = _test_env.load
_test_env.load = 0.4
_, _, _, _low_pen = _test_env._reliability_terms(0, 4, 6)
_test_env.load = 0.9
_, _, _, _high_pen = _test_env._reliability_terms(0, 4, 6)
_test_env.load = _saved_load
assert _high_pen > _low_pen, "High-load replica pressure penalty should be stronger."

_test_agent = PPOAgent(_test_env, minibatch_size=8, update_epochs=1, device="cpu")
_time = [0, (0, 0)]
_ep_time = [0, (0, 0)]
_hist = _test_agent.train(
    _time,
    _ep_time,
    csv_dir=_tmp,
    tasks_n=4,
    episodes=2,
    save_dir=os.path.join(_tmp, "logs"),
    policy_dir=os.path.join(_tmp, "policy"),
    log_last_k=1,
    seed=11,
    alphas=(0.3, 0.0, 0.55, 0.15),
)

assert len(_hist) == 2
_summary, _ = _test_agent.evaluate(episodes=2, deterministic=True)
assert _summary["capacity_violation_rate"] == 0.0

print("Smoke test passed.")
print("Observation dimension:", _test_env.obs_dim)
print("Joint action count:", _test_env.n_actions)
print("Replica util semantics: standby (no automatic utilization multiplication)")
print("Low-load replica penalty:", _low_pen)
print("High-load replica penalty:", _high_pen)
print("Evaluation summary:", _summary)


## Experiment runner


In [ ]:

def run_experiment_sweep(
    cores=4,
    episodes=None,
    data_root=None,
    output_root=None,
    seed=123,
):
    """Train and evaluate PPO over ten utilization load points."""
    data_root = Path(data_root or PROJECT_ROOT / 'data' / 'generated')
    output_root = Path(output_root or PROJECT_ROOT / 'artifacts')
    if episodes is None:
        episodes = int(15_000 * (cores / 4))

    tasks_n = int(cores * 10)
    all_summaries = []

    for i in range(1, 11):
        csv_dir = os.path.join(
            data_root,
            f"tsp-{cores}c",
            f"limit_{(i * cores) / 10}",
            "1",
        )
        csv_files = [
            os.path.join(csv_dir, f"task{j}.csv")
            for j in range(1, tasks_n + 1)
        ]

        missing = [p for p in csv_files if not os.path.exists(p)]
        if missing:
            print(f"[skip] load point {i}: missing {len(missing)} task CSV(s) under {csv_dir}")
            continue

        print(f"\n===== load point {i}/10 | util_load={i * 0.1:.1f} =====")

        env = Envir(
            initial_voltage=1.1,
            initial_frequency=2e9,
            cores=cores,
            main_tasks=tasks_n,
            alpha=(0.3, 0.0, 0.55, 0.15),
            util_load=i * 0.1,
            capacity_limit=1.1,
            tsp=1.5,
            rep_max=10,
        )

        agent = PPOAgent(
            env,
            lr=3e-4,
            gamma=0.99,
            gae_lambda=0.95,
            clip_eps=0.2,
            vf_coef=0.5,
            ent_coef=0.01,
            update_epochs=4,
            minibatch_size=2048,
        )

        save_dir = os.path.join(output_root, "logs", str(i))
        policy_dir = os.path.join(output_root, "policy", str(i))
        info_dir = os.path.join(output_root, "info", str(i))
        os.makedirs(save_dir, exist_ok=True)
        os.makedirs(policy_dir, exist_ok=True)
        os.makedirs(info_dir, exist_ok=True)

        time_elapse = [0, (0, 0)]
        episode_elapse = [0, (0, 0)]

        rewards = agent.train(
            time_elapse,
            episode_elapse,
            csv_dir=csv_dir,
            tasks_n=tasks_n,
            episodes=episodes,
            save_dir=save_dir,
            policy_dir=policy_dir,
            log_last_k=100,
            seed=seed,
            alphas=(0.3, 0.0, 0.55, 0.15),
        )

        eval_summary, eval_rows = agent.evaluate(
            episodes=20,
            seed=seed + 10_000,
            deterministic=True,
        )
        eval_rows.to_csv(os.path.join(info_dir, "evaluation_rows.csv"), index=False)

        timing_path = os.path.join(info_dir, f"info{i}.csv")
        with open(timing_path, mode="w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow(["Episode Worst", "Episode Avg", "Task Worst", "Task Avg"])
            writer.writerow([
                round(episode_elapse[0], 5),
                round(episode_elapse[1][0], 5),
                round(time_elapse[0], 5),
                round(time_elapse[1][0], 5),
            ])

        row = {
            "load_point": i,
            "util_load": i * 0.1,
            "train_final_reward": rewards[-1],
            **eval_summary,
        }
        all_summaries.append(row)
        pd.DataFrame([row]).to_csv(
            os.path.join(info_dir, "evaluation_summary.csv"),
            index=False,
        )

        print("Evaluation:", row)

    summary_df = pd.DataFrame(all_summaries)
    if not summary_df.empty:
        summary_df.to_csv(
            os.path.join(output_root, "ppo_all_loads_summary.csv"),
            index=False,
        )
    return summary_df


# Enable after running generate_task_sets.ipynb.
RUN_TRAINING = False

if RUN_TRAINING:
    sweep_summary = run_experiment_sweep(cores=4)
    display(sweep_summary)


## Evaluation metrics

Use fixed task CSVs, seeds, load points, hyperparameters, and reward weights. Report:

- reward per step;
- thermal violation rate;
- capacity violation rate;
- mean task/system reliability;
- mean replica count;
- maximum per-core utilization;
- generalization on held-out task sets.

Replica semantics check: for the same task/core/VF, changing `replica_n` must not change the normal utilization state. Its cost appears through reliability and the load-dependent replica-pressure penalty.
